<!-- open-in-badges -->
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/03_vision/notebooks/sesion_03_5_patches_vit.ipynb) [![Abrir en Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/03_vision/notebooks/sesion_03_5_patches_vit.ipynb) [![Abrir en Lightning Studio](https://pl-bolts-doc-images.s3.us-east-2.amazonaws.com/app-2/studio-badge.svg)](https://lightning.ai/new?repo_url=https%3A%2F%2Fgithub.com%2FEAFIT-IA%2Fsi7011-DeepLearning%2Fblob%2Fmain%2Fsessions%2F03_vision%2Fnotebooks%2Fsesion_03_5_patches_vit.ipynb)

# SI7011 · Sesión 03 · 5. De patches a Vision Transformers
**Juan David Martínez Vargas · Universidad EAFIT**

| Sección | Lo que cambia | Qué mirar |
|---|---|---|
| 1 | partir la imagen en patches, a mano y con `Conv2d` | que son la misma operación |
| 2 | self-attention en pocas líneas | pesos que suman 1; permutar tokens |
| 3 | un ViT pequeño desde cero frente a la CNN del notebook 3 | quién gana con 10 000 imágenes |
| 4 | el mismo ViT sin embedding posicional | lo que aporta saber dónde estaba cada patch |
| 5 | un ViT preentrenado en ImageNet, con linear probe | lo que cambia el preentrenamiento |
| 6 | el mapa de atención del ViT preentrenado | a qué patches mira el token `[CLS]` |

La atención se ve aquí solo lo necesario para el ViT; sus detalles (varias cabezas, máscaras, costo) son tema de la sesión 05. **Usa GPU** (Colab o Kaggle): unos 15 minutos. En CPU la sección 5 usa menos imágenes.

In [ ]:
import json, math, time
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import torch
import torch.nn.functional as F
from torch import nn
from torchvision import datasets
from torchvision.models import vit_b_16, ViT_B_16_Weights

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SMALL = device.type == 'cpu'
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
int_epochs = lambda: plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))
print('PyTorch', torch.__version__, '· device', device)

## Datos: CIFAR-10, como en los notebooks 1–3

In [ ]:
train_set = datasets.CIFAR10('data', train=True, download=True)
test_set = datasets.CIFAR10('data', train=False, download=True)
CLASSES = train_set.classes
to_tensor = lambda a: torch.from_numpy(a).permute(0, 3, 1, 2).float() / 255
X = to_tensor(train_set.data); y = torch.tensor(train_set.targets)
perm = torch.randperm(len(X), generator=torch.Generator().manual_seed(0))
R_train, y_train = X[perm[:10_000]], y[perm[:10_000]]
R_val, y_val = X[perm[45_000:]], y[perm[45_000:]]
R_test, y_test = to_tensor(test_set.data), torch.tensor(test_set.targets)

mean, std = R_train.mean(dim=(0, 2, 3)), R_train.std(dim=(0, 2, 3))
norm = lambda X: (X - mean.view(1, 3, 1, 1)) / std.view(1, 3, 1, 1)
X_train, X_val, X_test = norm(R_train), norm(R_val), norm(R_test)
X_train.shape

## 1. La imagen como secuencia de patches
Con patches de 4×4, una imagen de 32×32 se parte en $8 \times 8 = 64$ patches de $4 \cdot 4 \cdot 3 = 48$ números (en la diapositiva, 224×224 con patches de 16×16 da 196). Primero a mano, con `unfold`:

In [ ]:
P, d = 4, 64                                             # tamaño del patch y del token
x = X_train[:2]                                          # 2 × 3 × 32 × 32
patches = F.unfold(x, kernel_size=P, stride=P)           # 2 × 48 × 64: cada columna es un patch aplanado
patches = patches.transpose(1, 2)                        # 2 × 64 × 48: una fila por patch
torch.manual_seed(0)
W_E = nn.Linear(3 * P * P, d)                            # la misma matriz para todos los patches
tokens_a_mano = W_E(patches)
tokens_a_mano.shape

In [ ]:
conv = nn.Conv2d(3, d, kernel_size=P, stride=P)          # diapositiva Patch embedding is a convolution
with torch.no_grad():
    conv.weight.copy_(W_E.weight.view(d, 3, P, P)); conv.bias.copy_(W_E.bias)
tokens_conv = conv(x).flatten(2).transpose(1, 2)         # 2 × d × 8 × 8 → 2 × 64 × d
print('¿iguales?', torch.allclose(tokens_a_mano, tokens_conv, atol=1e-5))

In [ ]:
fig, axes = plt.subplots(8, 8, figsize=(3.4, 3.4))
for k, ax in enumerate(axes.flat):
    i, j = divmod(k, 8)
    ax.imshow(R_train[0, :, P * i:P * (i + 1), P * j:P * (j + 1)].permute(1, 2, 0)); ax.axis('off')
plt.suptitle(f'{CLASSES[y_train[0]]}: 64 patches de 4×4', fontsize=9); plt.show()

## 2. Self-attention en pocas líneas
Cada token se reemplaza por un **promedio ponderado de todos los tokens**. Los pesos salen de comparar *consultas* con *claves* (diapositiva *Self-attention, in one slide*):
$$\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V,\qquad Q = XW_Q,\ K = XW_K,\ V = XW_V$$

In [ ]:
def attention(X, W_Q, W_K, W_V):
    Q, K, V = X @ W_Q, X @ W_K, X @ W_V
    A = torch.softmax(Q @ K.transpose(-2, -1) / math.sqrt(K.shape[-1]), dim=-1)   # pesos: tokens × tokens
    return A @ V, A

torch.manual_seed(0)
W_Q, W_K, W_V = (torch.randn(d, d) / math.sqrt(d) for _ in range(3))
with torch.no_grad():
    out, A = attention(tokens_conv[:1], W_Q, W_K, W_V)
print('salida', tuple(out.shape), '· pesos', tuple(A.shape), '· cada fila suma', A[0].sum(-1)[:3])

Cada uno de los 64 tokens mira a los 64, incluido él mismo, en **una sola capa**. Ahora permutamos los tokens. Si la atención solo compara contenidos, la salida debería quedar permutada de la misma manera.

In [ ]:
sigma = torch.randperm(64)
with torch.no_grad():
    out_perm, _ = attention(tokens_conv[:1, sigma], W_Q, W_K, W_V)
print('error máximo entre attention(permutar) y permutar(attention):', (out_perm - out[:, sigma]).abs().max().item())

El error es cero, salvo redondeo: la atención no sabe dónde estaba cada patch. Es el experimento de permutación del notebook 1, ahora visto desde adentro. Por eso el ViT suma un **embedding posicional** aprendido a cada token (diapositiva *Position must be added*).

## Herramientas: el loop del notebook 3
Es el mismo loop con `augment`, con dos agregados del notebook 4: AdamW y un `warmup_cosine`. Los Transformers casi siempre se entrenan con *warmup*.

In [ ]:
loss_fn = nn.CrossEntropyLoss()

def augment(xb, pad=4):
    flip = torch.rand(len(xb), device=xb.device) < 0.5
    xb = torch.where(flip.view(-1, 1, 1, 1), xb.flip(3), xb)
    padded = F.pad(xb, (pad, pad, pad, pad), mode='reflect')
    ij = torch.randint(0, 2 * pad + 1, (len(xb), 2))
    return torch.stack([padded[k, :, i:i + 32, j:j + 32] for k, (i, j) in enumerate(ij.tolist())])

def warmup_cosine(opt, steps_per_epoch, epochs):
    total, warm = steps_per_epoch * epochs, steps_per_epoch
    f = lambda t: (t + 1) / warm if t < warm else 0.5 * (1 + math.cos(math.pi * (t - warm) / max(1, total - warm)))
    return torch.optim.lr_scheduler.LambdaLR(opt, f)

@torch.no_grad()
def evaluate(model, X, y, chunk=1000):
    model.eval()
    logits = torch.cat([model(X[i:i + chunk].to(device)).cpu() for i in range(0, len(X), chunk)])
    return loss_fn(logits, y).item(), (logits.argmax(1) == y).float().mean().item()

def fit(model, epochs=20, lr=1e-3, batch_size=128):
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    sched = warmup_cosine(opt, math.ceil(len(X_train) / batch_size), epochs)
    hist = {'loss': [], 'val_acc': []}
    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        perm, total = torch.randperm(len(X_train)), 0.0
        for i in range(0, len(X_train), batch_size):
            b = perm[i:i + batch_size]
            loss = loss_fn(model(augment(X_train[b].to(device))), y_train[b].to(device))
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            total += loss.item() * len(b)
        hist['loss'].append(total / len(X_train)); hist['val_acc'].append(evaluate(model, X_val, y_val)[1])
    print(f"época {epochs}: loss {hist['loss'][-1]:.3f} · val_acc {hist['val_acc'][-1]:.3f} · {time.time() - t0:.0f} s")
    return hist

def plot_curves(hists, key='val_acc', title=''):
    for name, h in hists.items():
        plt.plot(range(1, len(h[key]) + 1), h[key], label=name)
    plt.xlabel('época'); plt.ylabel(key); plt.title(title); plt.legend(); int_epochs(); plt.show()

n_params = lambda m: sum(p.numel() for p in m.parameters())

## 3. Un ViT pequeño desde cero
Las piezas son las de la diapositiva *The Vision Transformer*: patches → $W_E$ → token `[CLS]` → $+ p_i$ → $D$ bloques pre-norm (LayerNorm, atención, residual, LayerNorm, MLP, residual) → cabeza sobre `[CLS]`. Los bloques los da `nn.TransformerEncoderLayer` con `norm_first=True`.

In [ ]:
class ViT(nn.Module):
    def __init__(self, patch=4, d=128, depth=6, heads=4, n_classes=10, img=32, pos=True):
        super().__init__()
        n_tokens = (img // patch) ** 2
        self.embed = nn.Conv2d(3, d, kernel_size=patch, stride=patch)                 # W_E
        self.cls = nn.Parameter(torch.zeros(1, 1, d))
        self.pos = nn.Parameter(torch.randn(1, n_tokens + 1, d) * 0.02, requires_grad=pos)
        if not pos:
            nn.init.zeros_(self.pos)                                                    # sin posición
        layer = nn.TransformerEncoderLayer(d, heads, dim_feedforward=2 * d, dropout=0.0,
                                           activation='gelu', batch_first=True, norm_first=True)
        self.blocks = nn.TransformerEncoder(layer, depth, enable_nested_tensor=False)
        self.norm, self.head = nn.LayerNorm(d), nn.Linear(d, n_classes)

    def forward(self, x):
        t = self.embed(x).flatten(2).transpose(1, 2)                                    # N × 64 × d
        t = torch.cat([self.cls.expand(len(t), -1, -1), t], dim=1) + self.pos           # N × 65 × d
        return self.head(self.norm(self.blocks(t))[:, 0])                               # salida de [CLS]

torch.manual_seed(0)
vit = ViT()
print(f'ViT: {n_params(vit):,} parámetros')

Para comparar, la red con bloques VGG y BatchNorm del notebook 3, entrenada con el mismo loop, el mismo optimizador y las mismas épocas.

In [ ]:
def conv_bn_relu(c_in, c_out):
    return [nn.Conv2d(c_in, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU()]

def vgg_net(widths=(16, 32, 64)):
    blocks, c = [], 3
    for w in widths:
        blocks.append(nn.Sequential(*conv_bn_relu(c, w), *conv_bn_relu(w, w), nn.MaxPool2d(2))); c = w
    return nn.Sequential(*blocks, nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(c, 10))

torch.manual_seed(0)
cnn = vgg_net()
print(f'CNN: {n_params(cnn):,} parámetros')
h_cnn = fit(cnn)
torch.manual_seed(0)
h_vit = fit(vit)
plot_curves({'CNN (bloques VGG + BN)': h_cnn, 'ViT pequeño': h_vit}, title='Accuracy de validación')

Con 10 000 imágenes la CNN gana con clara ventaja, aunque el ViT tiene más parámetros. La CNN trae de fábrica la localidad y la equivarianza a traslaciones; el ViT tiene que aprenderlas de los datos, y 10 000 imágenes no alcanzan (diapositiva *Inductive bias versus data*).

## 4. El mismo ViT sin posición
**Cambia un argumento:** `ViT(pos=False)`. El embedding posicional queda en cero y no se entrena: el modelo ve una **bolsa de patches**, sin saber dónde estaba cada uno.

In [ ]:
torch.manual_seed(0)
h_vit_nopos = fit(ViT(pos=False))
plot_curves({'ViT pequeño': h_vit, 'ViT sin posición': h_vit_nopos}, title='Accuracy de validación')

Sin posición la accuracy baja: el modelo todavía reconoce texturas y colores, pero no su disposición. La diferencia suele ser menor de lo que uno esperaría, porque muchas clases de CIFAR-10 se distinguen por la textura.

## 5. Un ViT preentrenado en ImageNet
**Cambia los pesos:** ViT-B/16 de `torchvision`, entrenado en ImageNet, con 86 millones de parámetros y la arquitectura de la diapositiva ($d = 768$, $D = 12$). Como en el notebook 4, quitamos la cabeza y hacemos una linear probe sobre la salida de `[CLS]`.

El modelo espera imágenes de 224×224 normalizadas con la media de ImageNet, así que agrandamos cada imagen de 32×32 (se ve borrosa, pero funciona). En CPU usamos menos imágenes para que termine en un tiempo razonable.

In [ ]:
vit_weights = ViT_B_16_Weights.IMAGENET1K_V1
preset = vit_weights.transforms()
IM_MEAN, IM_STD = torch.tensor(preset.mean, device=device).view(1, 3, 1, 1), torch.tensor(preset.std, device=device).view(1, 3, 1, 1)
big = vit_b_16(weights=vit_weights)
big.heads = nn.Identity()                                    # salida: el token [CLS] final, 768 números
big = big.to(device).eval()
to_224 = lambda r: (F.interpolate(r.to(device), size=224, mode='bilinear', align_corners=False) - IM_MEAN) / IM_STD

@torch.no_grad()
def vit_features(R, chunk=128):
    with torch.autocast(device.type, enabled=device.type == 'cuda'):
        return torch.cat([big(to_224(R[i:i + chunk])).float().cpu() for i in range(0, len(R), chunk)])

N_PROBE, N_PROBE_VAL = (2000 if SMALL else 10_000), (1000 if SMALL else 5_000)
t0 = time.time()
G_train, G_val = vit_features(R_train[:N_PROBE]), vit_features(R_val[:N_PROBE_VAL])
print(f'features {tuple(G_train.shape)} en {time.time() - t0:.0f} s')

In [ ]:
torch.manual_seed(0)
head = nn.Linear(768, 10)
opt = torch.optim.Adam(head.parameters(), lr=1e-3, weight_decay=1e-4)
for _ in range(300):
    loss = loss_fn(head(G_train), y_train[:N_PROBE]); opt.zero_grad(); loss.backward(); opt.step()
with torch.no_grad():
    acc_big = (head(G_val).argmax(1) == y_val[:N_PROBE_VAL]).float().mean().item()
print(f'ViT-B/16 preentrenado + linear probe · val_acc {acc_big:.3f} ({N_PROBE} imágenes de entrenamiento)')

In [ ]:
resultados = {'cnn': max(h_cnn['val_acc']), 'vit': max(h_vit['val_acc']),
              'vit_sin_posicion': max(h_vit_nopos['val_acc']), 'vit_preentrenado_probe': acc_big}
json.dump(resultados, open('resultados_vit.json', 'w'), indent=1)
plt.figure(figsize=(6, 2.6))
plt.barh(list(resultados)[::-1], list(resultados.values())[::-1], color=['#d9631e', '#8a94a3', '#1f6fb4', '#167d78'])
plt.xlabel('accuracy de validación'); plt.xlim(0, 1); plt.show()

La misma familia de arquitecturas pasa del último lugar al primero. Lo que cambia es que el ViT preentrenado ya aprendió de ImageNet, con más de un millón de imágenes, lo que la CNN trae como supuesto. Es la figura de la diapositiva *Inductive bias versus data*, con nuestros números.

## 6. A qué mira el token `[CLS]`
`torchvision` no devuelve los pesos de atención, así que los recalculamos para el último bloque. Con un *hook* tomamos la entrada de su atención y volvemos a llamar a la capa con `need_weights=True`. La fila de `[CLS]` dice cuánto mira a cada uno de los $14 \times 14 = 196$ patches.

In [ ]:
last = big.encoder.layers[-1]
captured = {}
h = last.ln_1.register_forward_hook(lambda m, i, o: captured.__setitem__('x', o))
idx = torch.tensor([3, 11, 25, 42])
with torch.no_grad():
    big(to_224(R_val[idx]))
    _, A = last.self_attention(captured['x'], captured['x'], captured['x'], need_weights=True, average_attn_weights=True)
h.remove()
cls_map = A[:, 0, 1:].reshape(-1, 14, 14).float().cpu()       # atención de [CLS] a cada patch

fig, axes = plt.subplots(2, 4, figsize=(9, 4.4))
for k in range(len(idx)):
    axes[0, k].imshow(R_val[idx[k]].permute(1, 2, 0)); axes[0, k].set_title(CLASSES[y_val[idx[k]]], fontsize=8)
    axes[1, k].imshow(R_val[idx[k]].permute(1, 2, 0), extent=(0, 14, 14, 0))
    axes[1, k].imshow(cls_map[k], cmap='inferno', alpha=0.6, extent=(0, 14, 14, 0))
for ax in axes.flat: ax.axis('off')
axes[1, 0].set_title('atención de [CLS]', fontsize=8, loc='left')
plt.show()

Los pesos dependen del **contenido**: cambian de imagen a imagen y suelen concentrarse en el objeto. Una CNN, en cambio, mezcla siempre el mismo vecindario fijo. En la sesión 05 veremos las varias cabezas de atención y su costo con muchos tokens.

## Evaluación en prueba
Una sola vez, con el mejor de los modelos entrenados desde cero según validación.

In [ ]:
cands = {'CNN': (cnn, h_cnn), 'ViT pequeño': (vit, h_vit)}
name = max(cands, key=lambda k: max(cands[k][1]['val_acc']))
print(f'{name} · prueba: acc {evaluate(cands[name][0], X_test, y_test)[1]:.3f}')

## Tu turno · 10 minutos
1. Cambia el tamaño de patch a 8 (16 tokens) y luego a 2 (256 tokens). ¿Qué pasa con la accuracy y con el tiempo por época? ¿Por qué el costo crece tanto con patches pequeños?
2. Quita el token `[CLS]` y usa el promedio de todos los tokens antes de la cabeza. ¿Cambia algo?
3. Entrena el ViT pequeño con las 45 000 imágenes de entrenamiento (`perm[:45_000]`) en GPU. ¿Se acorta la distancia con la CNN?
4. En la sección 6, mira la atención de un patch en vez de la de `[CLS]`: usa la fila `A[:, 1 + 7 * 14 + 7, 1:]`, el patch del centro.

**Para pensar.** El ViT pequeño y el ViT-B/16 usan la misma idea. ¿Qué tendrías que tener, en datos o en cómputo, para entrenar el segundo desde cero, y qué conclusión sacas sobre cuándo usar cada arquitectura?

In [ ]:
# Experimento 1: patch = 8

### Referencias
- Dosovitskiy et al. (2021), *An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale*, ICLR.
- Vaswani et al. (2017), *Attention Is All You Need*, NeurIPS.
- Liu et al. (2022), *A ConvNet for the 2020s*, CVPR (ConvNeXt).
- Abnar y Zuidema (2020), *Quantifying Attention Flow in Transformers*, ACL (por qué un mapa de atención es una pista, no una explicación).